# Carga de datos

In [34]:
import pandas as pd
from os import path

In [35]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [36]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

In [298]:
ruta = path.join("datos_base.csv")

datos_base = pd.read_csv(
    ruta,
    sep=";",          
    quotechar='"',     
    engine="python",   
    encoding="utf-8",)

df = datos_base.copy()


In [299]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 915 entries, 0 to 914
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   correlativo                   915 non-null    int64 
 1   id_paciente                   915 non-null    int64 
 2   fecha                         915 non-null    object
 3   or_suite                      915 non-null    int64 
 4   servicio                      915 non-null    object
 5   codigo_cpt                    915 non-null    int64 
 6   descripcion                   915 non-null    object
 7   duracion_agendada             915 non-null    int64 
 8   hora_agendada                 915 non-null    object
 9   paciente_ingresa              915 non-null    object
 10  inicio_intervencion           915 non-null    object
 11  fin_intervencion              915 non-null    object
 12  paciente_sale                 915 non-null    object
 13  prioridad_paciente  

In [300]:
df["fecha"] = pd.to_datetime(datos_base["fecha"], format = "%d/%m/%Y")
df["hora_agendada"] = pd.to_datetime(datos_base["hora_agendada"], format = "%d/%m/%Y %H:%M")
df["paciente_ingresa"] = pd.to_datetime(datos_base["paciente_ingresa"], format = "%d/%m/%Y %H:%M")
df["inicio_intervencion"] = pd.to_datetime(datos_base["inicio_intervencion"], format = "%d/%m/%Y %H:%M")
df["fin_intervencion"] = pd.to_datetime(datos_base["fin_intervencion"], format = "%d/%m/%Y %H:%M")
df["paciente_sale"] = pd.to_datetime(datos_base["paciente_sale"], format = "%d/%m/%Y %H:%M")

In [301]:
df["dia_semana"] = df["fecha"].dt.dayofweek  # 0=Lunes
df["hora_inicio_prog"] = df["hora_agendada"].dt.hour

# Bloque del día (1 = tarde, 0 = mañana)
def bloque_dia(hora):
    if hora < 12:
        return 0
    elif hora < 19:
        return 1


df["bloque_dia"] = df["hora_inicio_prog"].apply(bloque_dia)


In [302]:
variables_cualitativas = ["servicio", "descripcion"]

encoder = OneHotEncoder(sparse_output=False)

encoded_cualitativas = encoder.fit_transform(df[variables_cualitativas])

columnas_encoded = encoder.get_feature_names_out(variables_cualitativas)

In [303]:

df_encoded = pd.DataFrame(
    encoded_cualitativas,
    columns=columnas_encoded,
    index=df.index)



In [304]:
df.head()

,correlativo,id_paciente,fecha,or_suite,servicio,codigo_cpt,descripcion,duracion_agendada,hora_agendada,paciente_ingresa,inicio_intervencion,fin_intervencion,paciente_sale,prioridad_paciente,dias_permanencia_programados,dias_permanencia_efectivos,dia_semana,hora_inicio_prog,bloque_dia
0,706,10707,2022-01-02,5,ENT,42826,Tonsillectomy,60,2022-01-02 07:00:00,2022-01-02 07:04:00,2022-01-02 07:24:00,2022-01-02 07:49:00,2022-01-02 08:00:00,5,1,1,6,7,0
1,707,10708,2022-01-02,5,ENT,30520,Septoplasty,90,2022-01-02 08:15:00,2022-01-02 08:30:00,2022-01-02 08:48:00,2022-01-02 09:40:00,2022-01-02 09:53:00,1,0,0,6,8,0
2,708,10709,2022-01-02,5,ENT,30520,Septoplasty,90,2022-01-02 10:00:00,2022-01-02 10:20:00,2022-01-02 10:45:00,2022-01-02 11:38:00,2022-01-02 11:49:00,3,0,0,6,10,0
3,709,10710,2022-01-02,5,ENT,42826,Tonsillectomy,60,2022-01-02 11:45:00,2022-01-02 12:30:00,2022-01-02 12:54:00,2022-01-02 13:26:00,2022-01-02 13:38:00,4,1,1,6,11,0
4,710,10711,2022-01-02,5,ENT,42826,Tonsillectomy,60,2022-01-02 13:00:00,2022-01-02 14:05:00,2022-01-02 14:28:00,2022-01-02 15:00:00,2022-01-02 15:10:00,1,1,1,6,13,1


In [305]:
df["duracion_efectiva_intervencion"] = (df["fin_intervencion"] - df["inicio_intervencion"]).dt.total_seconds() / 60
df["tiempo_total_pabellon"] = (df["paciente_sale"] - df["paciente_ingresa"]).dt.total_seconds() / 60

In [306]:
# ORDENAR CORRECTAMENTE
df = df.sort_values(
    ["or_suite", "fecha", "inicio_intervencion"]
)

# FIN PREVIO  (dentro del mismo OR y día)
df["fin_previo"] = (
    df
    .groupby(["or_suite", "fecha"])["paciente_sale"]
    .shift(1)
)

# REEMPLAZAR FIN_PREVIO 
df["fin_previo"] = df["fin_previo"].fillna(
    df["paciente_ingresa"]
)

# TIEMPO ENTRE CIRUGÍAS (minutos)
df["tiempo_entre_cirugias"] = (
    df["paciente_ingresa"] - df["fin_previo"]
).dt.total_seconds() / 60

df["tiempo_entre_cirugias"] = df["tiempo_entre_cirugias"].fillna(0)


In [307]:
cirugia0 = df[df["tiempo_entre_cirugias"] < 0]

print(cirugia0[["or_suite", "fecha", "paciente_ingresa", "fin_previo", "tiempo_entre_cirugias"]])

     or_suite      fecha    paciente_ingresa          fin_previo  \
278         3 2022-07-03 2022-07-03 12:12:00 2022-07-03 12:30:00   
280         3 2022-07-03 2022-07-03 12:55:00 2022-07-03 13:22:00   
282         3 2022-07-03 2022-07-03 13:50:00 2022-07-03 13:59:00   
283         3 2022-07-03 2022-07-03 14:20:00 2022-07-03 14:29:00   
317         3 2022-11-02 2022-11-02 07:03:00 2022-11-02 07:23:00   
324         3 2022-11-02 2022-11-02 12:55:00 2022-11-02 13:22:00   
326         3 2022-11-02 2022-11-02 13:50:00 2022-11-02 13:59:00   
327         3 2022-11-02 2022-11-02 14:20:00 2022-11-02 14:29:00   

     tiempo_entre_cirugias  
278                  -18.0  
280                  -27.0  
282                   -9.0  
283                   -9.0  
317                  -20.0  
324                  -27.0  
326                   -9.0  
327                   -9.0  


In [308]:

for col in df.columns:
    if df[col].dtype == "bool":
        df[col] = df[col].astype(int)
    
    elif df[col].dtype == "float64":
        df[col] = df[col].astype(int)


# Prediccion duraciones

In [309]:
def evaluar_modelo(y_test, y_pred, nombre):
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"\n{nombre}:")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.4f}")
    
    return mae, rmse, r2

In [310]:
def entrenar_xgboost(X, y, nombre_objetivo):


    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    modelo = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    modelo.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    y_pred = modelo.predict(X_test)
    metricas = evaluar_modelo(
        y_test, y_pred,
        f"{nombre_objetivo} - XGBoost"
    )

    return {
        "modelo": modelo,
        "metricas": metricas,
        "X_test": X_test,
        "y_test": y_test,
        "y_pred": y_pred
    }


In [311]:

features = ["or_suite",
    "duracion_agendada",
    "prioridad_paciente",
    # one-hot ya existentes:
    *[c for c in df.columns if c.startswith("servicio_")],
    *[c for c in df.columns if c.startswith("descripcion_")]]


In [312]:

X = df[features]
y = df["duracion_efectiva_intervencion"]

resultado_xgb = entrenar_xgboost(
    X, y, "Duración efectiva intervención"
)

modelo_final = resultado_xgb["modelo"]



Duración efectiva intervención - XGBoost:
MAE: 4.35
RMSE: 6.47
R²: 0.9356


In [393]:
ruta_2 = path.join("lista_de_espera.csv")
lista_espera = pd.read_csv(
    ruta_2,
    sep=";",          
    quotechar='"',     
    engine="python",   
    encoding="utf-8",)
df_lista_espera = lista_espera.copy()


In [394]:
def preparar_lista_espera(df, columnas_modelo):
    X = pd.DataFrame(0, index=df.index, columns=columnas_modelo)

    # Variables numéricas
    X["or_suite"] = df["OR Suite"]
    X["duracion_agendada"] = df["Duración agendada (min)"]
    X["prioridad_paciente"] = df["Prioridad de paciente"]

    # Variables categóricas → one-hot
    for i, fila in df.iterrows():
        servicio_col = f"servicio_{fila['Servicio']}"
        descripcion_col = f"descripcion_{fila['Descripción']}"

        if servicio_col in X.columns:
            X.loc[i, servicio_col] = 1

        if descripcion_col in X.columns:
            X.loc[i, descripcion_col] = 1

    return X

In [395]:

X_lista = preparar_lista_espera(df_lista_espera, modelo_final.get_booster().feature_names)

df_lista_espera["Duración efectiva estimada (min)"] = modelo_final.predict(X_lista)

df_lista_espera["Duración efectiva estimada (min)"] = df_lista_espera["Duración efectiva estimada (min)"].round().astype(int)


In [396]:
df_lista_espera.head(20)

,Correlativo,ID paciente,Fecha,OR Suite,Servicio,Código CPT,Descripción,Duración agendada (min),Prioridad de paciente,Duración efectiva estimada (min)
0,273,10274,NaN,1,Podiatry,28296,Bunionectomy with distal osteotomy,120,5,84
1,274,10275,NaN,1,Podiatry,28289,Hallux rigidus correction with cheilectomy,60,4,40
2,275,10276,NaN,1,Podiatry,28296,Bunionectomy with distal osteotomy,120,5,84
3,276,10277,NaN,1,Podiatry,28296,Bunionectomy with distal osteotomy,120,5,84
4,277,10278,NaN,2,Orthopedics,29877,"Arthroscopy, knee, surgical",60,2,36
5,278,10279,NaN,2,Orthopedics,29877,"Arthroscopy, knee, surgical",60,3,34
6,279,10280,NaN,2,Orthopedics,29877,"Arthroscopy, knee, surgical",60,4,34
7,280,10281,NaN,2,Orthopedics,29877,"Arthroscopy, knee, surgical",60,1,36
8,281,10282,NaN,3,Ophthalmology,66982,Extracapsular cataract removal,45,3,17
9,282,10283,NaN,3,Ophthalmology,66982,Extracapsular cataract removal,45,2,17


# Duracion Real a priori

vamos a asignar probabilidades a que la duracion real sea mayor

Duración real = Duración estimada + Error aleatorio

In [397]:

def construir_distribucion_errores(y_real, y_pred):
    errores = y_real - y_pred
    return errores


In [398]:

errores_historicos = construir_distribucion_errores(
    resultado_xgb["y_test"],
    resultado_xgb["y_pred"]
)


In [399]:
def simular_duracion_real(
    duracion_estimada,
    errores_historicos,
    n_simulaciones=1000,
    minimo=5
):
    errores_sampleados = np.random.choice(
        errores_historicos,
        size=n_simulaciones,
        replace=True
    )

    duraciones_simuladas = duracion_estimada + errores_sampleados
    duraciones_simuladas = np.clip(duraciones_simuladas, minimo, None)

    return duraciones_simuladas

In [400]:
def simular_fila(
    duracion_estimada,
    errores_historicos,
    n_simulaciones=5000,
    minimo=5
):
    errores_sampleados = np.random.choice(
        errores_historicos,
        size=n_simulaciones,
        replace=True
    )

    duraciones = duracion_estimada + errores_sampleados
    duraciones = np.clip(duraciones, minimo, None)

    return {
        "Duración p50 (min)": np.percentile(duraciones, 50) + 5,
        "Duración p75 (min)": np.percentile(duraciones, 75) + 10,
        "Duración p90 (min)": np.percentile(duraciones, 90) + 15,
        "Duración p95 (min)": np.percentile(duraciones, 95) + 20
    }


In [401]:

resultados_simulados = df_lista_espera[
    "Duración efectiva estimada (min)"
].apply(
    lambda x: simular_fila(x, errores_historicos)
)

df_simulacion = pd.DataFrame(list(resultados_simulados))


In [402]:

df_lista_espera = pd.concat(
    [df_lista_espera.reset_index(drop=True), df_simulacion],
    axis=1
)


In [403]:
cols = [
    "Duración p50 (min)",
    "Duración p75 (min)",
    "Duración p90 (min)",
    "Duración p95 (min)"
]

df_lista_espera[cols] = (
    df_lista_espera[cols]
        .round()
        .astype(int)
)


In [404]:
df_lista_espera.head(20)

,Correlativo,ID paciente,Fecha,OR Suite,Servicio,Código CPT,Descripción,Duración agendada (min),Prioridad de paciente,Duración efectiva estimada (min),Duración p50 (min),Duración p75 (min),Duración p90 (min),Duración p95 (min)
0,273,10274,NaN,1,Podiatry,28296,Bunionectomy with distal osteotomy,120,5,84,89,96,104,114
1,274,10275,NaN,1,Podiatry,28289,Hallux rigidus correction with cheilectomy,60,4,40,45,52,60,70
2,275,10276,NaN,1,Podiatry,28296,Bunionectomy with distal osteotomy,120,5,84,89,96,104,114
3,276,10277,NaN,1,Podiatry,28296,Bunionectomy with distal osteotomy,120,5,84,89,96,105,114
4,277,10278,NaN,2,Orthopedics,29877,"Arthroscopy, knee, surgical",60,2,36,40,48,56,66
5,278,10279,NaN,2,Orthopedics,29877,"Arthroscopy, knee, surgical",60,3,34,39,46,55,64
6,279,10280,NaN,2,Orthopedics,29877,"Arthroscopy, knee, surgical",60,4,34,39,46,55,64
7,280,10281,NaN,2,Orthopedics,29877,"Arthroscopy, knee, surgical",60,1,36,41,48,57,66
8,281,10282,NaN,3,Ophthalmology,66982,Extracapsular cataract removal,45,3,17,22,29,37,47
9,282,10283,NaN,3,Ophthalmology,66982,Extracapsular cataract removal,45,2,17,21,29,38,47


# Camas

In [405]:
df["dias_permanencia_efectivos"] = (
    df["dias_permanencia_efectivos"].astype(int)
)


In [406]:

stats_por_desc = (
    df
    .groupby("descripcion")["dias_permanencia_efectivos"]
    .agg(
        min_dias="min",
        max_dias="max",
        n_casos="count"
    )
    .reset_index()
)


In [407]:

distribuciones_por_desc = {
    desc: grupo["dias_permanencia_efectivos"].values
    for desc, grupo in df.groupby("descripcion")
}


In [408]:
def resumen_empirico(distribucion, minimo):
    return {
        "Hosp p50 (días)": np.percentile(distribucion, 50),
        "Hosp p75 (días)": np.percentile(distribucion, 75),
        "Hosp p90 (días)": np.percentile(distribucion, 90),
        "Hosp p95 (días)": np.percentile(distribucion, 95),
        "Prob_+1_día": np.mean(distribucion >= minimo + 1),
        "Prob_+2_días": np.mean(distribucion >= minimo + 2),
        "Prob_+3_días": np.mean(distribucion >= minimo + 3),
    }


In [409]:
modelo_empirico = {}

for desc, distribucion in distribuciones_por_desc.items():
    minimo = distribucion.min()
    maximo = distribucion.max()

    modelo_empirico[desc] = {
        "min_dias": minimo,
        "max_dias": maximo,
        "n_casos": len(distribucion),
        **resumen_empirico(distribucion, minimo)
    }

In [410]:

df_modelo_empirico = (
    pd.DataFrame(modelo_empirico)
    .T
    .reset_index()
    .rename(columns={"index": "Descripción"})
)


In [411]:

df_lista_espera_hosp = df_lista_espera.merge(
    df_modelo_empirico,
    on="Descripción",
    how="left"
)


In [413]:
df_modelo_empirico.head(20)

,Descripción,min_dias,max_dias,n_casos,Hosp p50 (días),Hosp p75 (días),Hosp p90 (días),Hosp p95 (días),Prob_+1_día,Prob_+2_días,Prob_+3_días
0,AV fistula,1.0,8.0,38.0,4.0,5.00,6.0,7.15,0.947368,0.868421,0.657895
1,"Adjacent tissue transfer, eyelids, nose, ears,...",1.0,8.0,37.0,5.0,6.00,7.0,7.00,0.945946,0.864865,0.648649
2,"Arthroplasty, hip",2.0,8.0,10.0,4.5,6.00,7.1,7.55,0.900000,0.800000,0.500000
3,"Arthroplasty, knee, hinge prothesis",1.0,8.0,34.0,5.0,6.00,7.7,8.00,0.911765,0.823529,0.705882
4,"Arthroscopy, knee, surgical",1.0,8.0,49.0,5.0,6.00,7.2,8.00,0.959184,0.938776,0.734694
5,Bunionectomy with distal osteotomy,1.0,8.0,36.0,5.0,6.00,7.5,8.00,0.972222,0.861111,0.694444
6,"Carpal tunnel release, open",2.0,8.0,18.0,5.5,7.00,8.0,8.00,0.944444,0.833333,0.666667
7,Cervical biopsy,1.0,8.0,34.0,4.0,5.00,7.0,7.35,0.882353,0.705882,0.676471
8,"Correction, hammertoe",1.0,8.0,18.0,5.0,6.00,7.3,8.00,0.944444,0.777778,0.611111
9,Cryosurgery of the prostate gland,2.0,6.0,16.0,5.0,5.00,5.0,5.25,0.937500,0.812500,0.625000


# Caso base

In [425]:
def sample_duracion_intervencion(fila):
    p50 = fila["Duración p50 (min)"]
    p75 = fila["Duración p75 (min)"]
    p90 = fila["Duración p90 (min)"]
    p95 = fila["Duración p95 (min)"]

    u = np.random.rand()

    if u <= 0.50:
        return np.random.uniform(0.85 * p50, p50)
    elif u <= 0.75:
        return np.random.uniform(p50, p75)
    elif u <= 0.90:
        return np.random.uniform(p75, p90)
    elif u <= 0.95:
        return np.random.uniform(p90, p95)
    else:
        return np.random.uniform(p95, 1.10 * p95)




In [451]:
def sample_dias_permanencia_desde_percentiles(fila):
    minimo = fila["min_dias"]

    p50 = fila["Hosp p50 (días)"]
    p75 = fila["Hosp p75 (días)"]
    p90 = fila["Hosp p90 (días)"]
    p95 = fila["Hosp p95 (días)"]

    u = np.random.rand()

    if u <= 0.50:
        dias = np.random.randint(minimo, p50 + 1)
    elif u <= 0.75:
        dias = np.random.randint(p50, p75 + 1)
    elif u <= 0.90:
        dias = np.random.randint(p75, p90 + 1)
    elif u <= 0.95:
        dias = np.random.randint(p90, p95 + 1)
    else:
    # eventos muy raros
        dias = np.random.choice([p95, p95 + 1, p95 + 2], p=[0.7, 0.2, 0.1])


    # seguridad clínica
    return int(round(max(dias, minimo)))


In [452]:
def construir_caso_base_desde_percentiles(df_lista):
    filas = []

    for _, fila in df_lista.iterrows():

        duracion_efectiva = sample_duracion_intervencion(fila)
        dias_efectivos = sample_dias_permanencia_desde_percentiles(fila)

        filas.append({
            "Correlativo": fila["Correlativo"],
            "ID paciente": fila["ID paciente"],
            "Fecha": fila["Fecha"],
            "OR Suite": fila["OR Suite"],
            "Servicio": fila["Servicio"],
            "Código CPT": fila["Código CPT"],
            "Descripción": fila["Descripción"],
            "Duración agendada (min)": fila["Duración agendada (min)"],
            "Prioridad de paciente": fila["Prioridad de paciente"],
            "duracion_efectiva_intervencion": round(duracion_efectiva),
            "dias_permanencia_programados": fila["min_dias"],
            "dias_permanencia_efectivos": dias_efectivos
        })

    return pd.DataFrame(filas)

In [453]:
df_caso_base = construir_caso_base_desde_percentiles(df_lista_espera_hosp)

In [454]:
df_caso_base["dias_permanencia_programados"] = df_caso_base["dias_permanencia_programados"].apply(lambda x: int(x))

In [455]:
df_caso_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1257 entries, 0 to 1256
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Correlativo                     1257 non-null   int64  
 1   ID paciente                     1257 non-null   int64  
 2   Fecha                           0 non-null      float64
 3   OR Suite                        1257 non-null   int64  
 4   Servicio                        1257 non-null   object 
 5   Código CPT                      1257 non-null   int64  
 6   Descripción                     1257 non-null   object 
 7   Duración agendada (min)         1257 non-null   int64  
 8   Prioridad de paciente           1257 non-null   int64  
 9   duracion_efectiva_intervencion  1257 non-null   int64  
 10  dias_permanencia_programados    1257 non-null   int64  
 11  dias_permanencia_efectivos      1257 non-null   int64  
dtypes: float64(1), int64(9), object(2)

In [456]:
df_caso_base.to_csv(
    "caso_base/caso_base.csv",
    index=False,
    encoding="utf-8-sig"
)


In [457]:
df_caso_base["dias_permanencia_programados"].describe()

count    1257.000000
mean        0.983294
std         0.601649
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: dias_permanencia_programados, dtype: float64

In [458]:
df_caso_base["dias_permanencia_efectivos"].describe()

count    1257.000000
mean        3.456643
std         2.476624
min         0.000000
25%         1.000000
50%         4.000000
75%         5.000000
max        10.000000
Name: dias_permanencia_efectivos, dtype: float64

In [459]:
df_caso_base["dias_permanencia_efectivos"].value_counts().sort_index()

dias_permanencia_efectivos
0     216
1     167
2     107
3     106
4     177
5     174
6     154
7     115
8      31
9       9
10      1
Name: count, dtype: int64